# Notebook 2 — The Bioinformatics Toolkit
## Using Real Libraries Like Biopython and Pandas

**Course:** Introduction to Bioinformatics — HCT Cohort
**Instructor:** Dr. Norah
**Duration:** ~1.5 hours · 16 exercises

---

### What you'll learn
- Install and import Biopython
- Use Biopython's `Seq` object for DNA work
- Read and write FASTA files
- Build tables of biological data with pandas
- Visualize sequence data with matplotlib

### Why use libraries?
In the last notebook you wrote code to count bases, calculate GC content, and find motifs. Professional bioinformaticians don't rewrite this code every time — they use tested, trusted libraries. Today you'll learn the main ones.


---
## Exercise 1 — Install Biopython
Biopython isn't pre-installed in Colab, so let's install it.


In [ ]:
!pip install biopython --quiet


If the cell finished without errors, Biopython is ready to use.


---
## Exercise 2 — Your first `Seq` object
Biopython represents DNA, RNA, and protein sequences as `Seq` objects.
They're like strings but have bioinformatics superpowers.


In [ ]:
from Bio.Seq import Seq

dna = Seq("ATGGATTTATCTGCTCTT")
print("Sequence:", dna)
print("Type:    ", type(dna))
print("Length:  ", len(dna))


### ✏️ Your Turn
Create your own `Seq` object and print its length.


In [ ]:
my_seq = Seq("____")
print("My sequence length:", ____)


---
## Exercise 3 — `Seq` objects have familiar methods
`.count()`, `.upper()`, slicing — all the things you used in Notebook 1 still work.


In [ ]:
dna = Seq("ATGGATTTATCTGCTCTTCGCG")
print("Count of A:", dna.count("A"))
print("First 5:  ", dna[:5])
print("Last 5:   ", dna[-5:])


---
## Exercise 4 — Complement
The **complement** of a DNA strand pairs A↔T and G↔C.
It's the sequence on the other strand of the double helix.


In [ ]:
dna = Seq("ATGC")
print("Original:  ", dna)
print("Complement:", dna.complement())


### ✏️ Your Turn
What is the complement of `AATTCCGG`?


In [ ]:
seq = Seq("AATTCCGG")
print("Complement:", ____)  # Hint: .complement()


---
## Exercise 5 — Reverse complement
The **reverse complement** is what you get if you read the other strand 5' to 3'.
This is VERY common in bioinformatics — e.g., when finding genes on the opposite strand.


In [ ]:
dna = Seq("ATGCGATCG")
print("Original:          ", dna)
print("Complement:        ", dna.complement())
print("Reverse complement:", dna.reverse_complement())


---
## Exercise 6 — Transcription: DNA → RNA
Biopython has a `.transcribe()` method that converts DNA into RNA.
Remember: every T becomes a U.


In [ ]:
dna = Seq("ATGGATTTATCT")
rna = dna.transcribe()
print("DNA:", dna)
print("RNA:", rna)


### ✏️ Your Turn
Transcribe the BRCA1 partial coding sequence into RNA.


In [ ]:
brca1 = Seq("ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAA")
brca1_rna = ____  # Hint: .transcribe()
print(brca1_rna)


---
## Exercise 7 — Translation: RNA → Protein
Biopython's `.translate()` method turns a sequence into its protein.
Every 3 bases (codon) become one amino acid.


In [ ]:
dna = Seq("ATGGATTTATCTGCTCTTCGCGTT")
protein = dna.translate()
print("DNA:    ", dna)
print("Protein:", protein)


Notice that each amino acid is shown as a single letter (Met = M, Asp = D, etc.).

### ✏️ Your Turn
Translate the insulin gene's first few codons.


In [ ]:
insulin_dna = Seq("ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTG")
insulin_protein = ____
print(insulin_protein)


---
## Exercise 8 — Create and read a FASTA file
FASTA is the most common sequence file format. Let's write one, then read it back.


In [ ]:
# First, create a small FASTA file
fasta_text = """>BRCA1_partial Human BRCA1 (partial)
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA
>INS_partial Human insulin (partial)
ATGGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCT"""

with open("genes.fasta", "w") as f:
    f.write(fasta_text)

print("Saved genes.fasta")


In [ ]:
from Bio import SeqIO

for record in SeqIO.parse("genes.fasta", "fasta"):
    print("ID:         ", record.id)
    print("Description:", record.description)
    print("Length:     ", len(record.seq))
    print("First 30:   ", record.seq[:30])
    print("---")


---
## Exercise 9 — Analyze every record in a FASTA
Let's compute GC content for every sequence in the file.


In [ ]:
from Bio.SeqUtils import gc_fraction

for record in SeqIO.parse("genes.fasta", "fasta"):
    gc_pct = gc_fraction(record.seq) * 100
    print(f"{record.id}: length={len(record.seq)}  GC={gc_pct:.1f}%")


---
## Exercise 10 — pandas: turning sequences into a table
**pandas** is like Excel for Python. Let's build a table of our genes.


In [ ]:
import pandas as pd

rows = []
for record in SeqIO.parse("genes.fasta", "fasta"):
    rows.append({
        "id": record.id,
        "length": len(record.seq),
        "gc_percent": round(gc_fraction(record.seq) * 100, 2),
        "starts_with_ATG": str(record.seq).startswith("ATG")
    })

df = pd.DataFrame(rows)
df


---
## Exercise 11 — Filter the table
Select only rows that meet a condition.


In [ ]:
# Keep only sequences longer than 50 bases
long_ones = df[df["length"] > 50]
long_ones


### ✏️ Your Turn
Filter the table to keep only sequences with GC% greater than 40.


In [ ]:
high_gc = df[____]  # Hint: similar syntax to above
high_gc


---
## Exercise 12 — Summary statistics
`.describe()` gives quick statistics for all numeric columns.


In [ ]:
df.describe()


---
## Exercise 13 — Plot nucleotide composition
Let's make a bar chart showing how many of each base are in BRCA1.


In [ ]:
import matplotlib.pyplot as plt

brca1 = "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA"
bases = ["A", "T", "G", "C"]
counts = [brca1.count(b) for b in bases]

plt.figure(figsize=(6, 4))
plt.bar(bases, counts, color=["#028090", "#02C39A", "#05445E", "#00A896"])
plt.title("BRCA1 partial — nucleotide composition")
plt.xlabel("Base")
plt.ylabel("Count")
plt.show()


---
## Exercise 14 — Compare GC content across genes
Visualize the GC% of each sequence in our FASTA.


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(df["id"], df["gc_percent"], color="#028090")
plt.title("GC content by gene")
plt.ylabel("GC %")
plt.ylim(0, 100)
plt.show()


---
## Exercise 15 — Save a plot as a file
You can save any plot to a PNG file with `plt.savefig()`.


In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(bases, counts, color="#028090")
plt.title("BRCA1 nucleotide composition")
plt.xlabel("Base")
plt.ylabel("Count")
plt.savefig("brca1_plot.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved as brca1_plot.png")


---
## Exercise 16 — 🏆 Mini-project: A small multi-gene analysis
Put it all together. Below we define a FASTA with 4 genes. Your job:
1. Read the FASTA
2. Build a DataFrame with id, length, gc_percent
3. Sort by GC% descending
4. Plot a bar chart of GC% by gene

Fill in the blanks.


In [ ]:
fasta_multi = """>gene1 Example 1
ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA
>gene2 Example 2
GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC
>gene3 Example 3
AAAAATTTTTAAAAATTTTTAAAAATTTTTAAAAATTTTTAAAAATTTTTAAAAATTTTT
>gene4 Example 4
ATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGCATGC"""

with open("multi.fasta", "w") as f:
    f.write(fasta_multi)

# Step 1 + 2 — read FASTA and build DataFrame
rows = []
for record in SeqIO.parse("multi.fasta", "fasta"):
    rows.append({
        "id": record.id,
        "length": ____,
        "gc_percent": round(____ * 100, 2)
    })
mydf = pd.DataFrame(rows)

# Step 3 — sort
mydf = mydf.sort_values("gc_percent", ascending=False)
print(mydf)

# Step 4 — plot
plt.figure(figsize=(6, 4))
plt.bar(mydf["id"], mydf["gc_percent"], color="#02C39A")
plt.title("GC content across genes")
plt.ylabel("GC %")
plt.show()


---
## 🎉 Tremendous!

You just used the same tools that professional bioinformaticians use every day:

- ✅ **Biopython** for sequence objects, transcription, translation, FASTA parsing
- ✅ **pandas** for building and querying tables of biological data
- ✅ **matplotlib** for visualization

### Coming up next
In the final notebook, we'll trace a real gene (BRCA1) all the way through the **central dogma** — from DNA through RNA to Protein — and see how a single mutation can change everything.
